In [11]:
import pandas as pd
import numpy as np
import json
import warnings
from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore")

# ─── Пути ─────────────────────────────────────────────────────────────────────
DATA_DIR   = Path("data_sensors")
CONFIG_DIR = Path("configs")
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

# ─── Параметры очистки ────────────────────────────────────────────────────────
MAX_CAPACITY  = 150
MIN_DAYS      = 3
ASSUMED_SPEED = 15.0   # км/ч

# ─── Маршруты ─────────────────────────────────────────────────────────────────
ROUTES     = ["20", "48", "55"]
DIRECTIONS = ["Прямое", "Обратное"]

# ─── Префиксы глобальных stop_id ─────────────────────────────────────────────
# Чтобы остановки разных маршрутов не пересекались по ID случайно,
# каждому маршруту выделяем блок: маршрут 20 → 20001..20999, и т.д.
# Если две остановки физически одинаковые (пересечение маршрутов) —
# укажи их вручную в STOP_ID_OVERRIDES ниже.
ROUTE_STOP_PREFIX = {
    "20": 20000,
    "48": 48000,
    "55": 55000,
}

# Маппинг (route, direction, clean_stop_id) → global_stop_id
# для остановок, которые физически совпадают между маршрутами.
# Заполни вручную после первого прогона, когда увидишь названия остановок.
# Пример: остановка "Площадь Ленина" есть и в маршруте 20 (ост. 15),
#         и в маршруте 48 (ост. 8) → присваиваем им одинаковый global_stop_id.
#
# Формат: {("route", "direction", clean_stop_id_int): global_stop_id_int}
STOP_ID_OVERRIDES: dict = {
    # ("20", "Прямое",   15): 99001,
    # ("48", "Прямое",    8): 99001,
}
# ─── Расписание (будние дни) ───────────────────────────────────────────────────
# start/end — минуты от начала дня (05:29 = 329, итд)
# periods — [(from_min, to_min, interval_min), ...]
SCHEDULE_INTERVALS = {
    "20": {
        "start": 329,  # 05:29
        "end":   1380, # 23:00
        "periods": [(360, 600, 8), (600, 960, 11), (960, 1200, 9), (1200, 1380, 12)],
    },
    "48": {
        "start": 329,  # 05:29
        "end":   1425, # 23:45
        "periods": [(360, 600, 7), (600, 960, 10), (960, 1200, 9), (1200, 1425, 13)],
    },
    "55": {
        "start": 316,  # 05:16
        "end":   1394, # 23:14
        "periods": [(360, 600, 5), (600, 960, 8), (960, 1200, 5), (1200, 1394, 7)],
    },
}

DEPOT_TO_FIRST_STOP_MIN = 8  # минут от депо до первой остановки (константа)
MIN_REST_TIME_MIN        = 15 # минимальный отдых водителя после рейса


In [12]:
def load_all_data(data_path: Path) -> pd.DataFrame:
    all_files = list(data_path.glob("*.xlsx"))
    if not all_files:
        raise FileNotFoundError(f"Excel файлы не найдены в {data_path.resolve()}")

    COLUMNS = [
        "date", "route", "direction", "board_num", "trip_id",
        "stop_seq", "_stop_id", "stop_name", "arr_time",
        "boarded", "alighted", "in_cabin", "veh_type"
    ]

    df_list = []
    for f in all_files:
        df = pd.read_excel(f, skiprows=2, names=COLUMNS)
        df["_source_file"] = f.name
        df_list.append(df)

    df_raw = pd.concat(df_list, ignore_index=True)
    df_raw = df_raw.drop(columns=["_stop_id", "veh_type"])

    print(f"Загружено файлов: {len(all_files)}")
    print(f"Всего строк:      {len(df_raw):,}")
    return df_raw


df_raw = load_all_data(DATA_DIR)


Загружено файлов: 24
Всего строк:      331,316


In [13]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    n0 = len(df)

    for col in ["boarded", "alighted", "in_cabin", "stop_seq"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["arr_time"] = pd.to_datetime(df["arr_time"], errors="coerce")

    for col in ["route", "direction", "stop_name", "trip_id", "board_num"]:
        df[col] = df[col].astype(str).str.strip()

    df = df.dropna(subset=["arr_time", "route", "direction", "trip_id", "stop_seq"])
    print(f"  После dropna:               {len(df):,} ({len(df)/n0*100:.1f}%)")

    for col in ["boarded", "alighted", "in_cabin"]:
        df = df[df[col].between(0, MAX_CAPACITY)]
    print(f"  После отсечки >{MAX_CAPACITY}:          {len(df):,} ({len(df)/n0*100:.1f}%)")

    trip_sum = df.groupby("trip_id")[["boarded", "alighted", "in_cabin"]].sum()
    dead_trips = trip_sum[
        (trip_sum["boarded"] == 0) &
        (trip_sum["alighted"] == 0) &
        (trip_sum["in_cabin"] == 0)
    ].index
    df = df[~df["trip_id"].isin(dead_trips)]
    print(f"  После удаления мёртвых рейсов: {len(df):,} ({len(df)/n0*100:.1f}%)")

    df["hour"]      = df["arr_time"].dt.hour
    df["date_only"] = df["arr_time"].dt.date

    return df.reset_index(drop=True)


print("=== Очистка данных ===")
df_clean = clean_data(df_raw)

assert df_clean["in_cabin"].max() <= MAX_CAPACITY
assert df_clean["boarded"].max() <= MAX_CAPACITY
print(f"\n  in_cabin max: {df_clean['in_cabin'].max()}")
print(f"  boarded  max: {df_clean['boarded'].max()}")

=== Очистка данных ===
  После dropna:               284,379 (85.8%)
  После отсечки >150:          281,237 (84.9%)
  После удаления мёртвых рейсов: 80,762 (24.4%)

  in_cabin max: 150
  boarded  max: 130


In [14]:
def build_reference_route(df: pd.DataFrame, route: str, direction: str) -> pd.DataFrame:
    """
    Строит эталонную последовательность остановок.
    Добавляет global_stop_id с учётом STOP_ID_OVERRIDES.
    """
    d = df[(df["route"] == route) & (df["direction"] == direction)]

    if d.empty:
        print(f"  ПРЕДУПРЕЖДЕНИЕ: нет данных для маршрута {route} ({direction})")
        return pd.DataFrame(columns=["stop_seq", "stop_name", "clean_stop_id", "global_stop_id"])

    ref = (d.groupby("stop_seq")["stop_name"]
            .agg(lambda x: x.mode().iloc[0])
            .reset_index()
            .sort_values("stop_seq")
            .reset_index(drop=True))

    ref["clean_stop_id"] = np.arange(1, len(ref) + 1, dtype=int)

    # Назначаем глобальные ID
    prefix = ROUTE_STOP_PREFIX.get(route, 0)
    global_ids = []
    for _, row in ref.iterrows():
        key = (route, direction, int(row["clean_stop_id"]))
        if key in STOP_ID_OVERRIDES:
            global_ids.append(STOP_ID_OVERRIDES[key])
        else:
            global_ids.append(prefix + int(row["clean_stop_id"]))
    ref["global_stop_id"] = global_ids

    print(f"  Маршрут {route} ({direction}): {len(ref)} остановок "
          f"(seq {int(ref['stop_seq'].min())}–{int(ref['stop_seq'].max())}), "
          f"global_id {global_ids[0]}..{global_ids[-1]}")
    return ref


def attach_clean_stop_id(df: pd.DataFrame, ref: pd.DataFrame) -> pd.DataFrame:
    mapping = dict(zip(ref["stop_seq"], ref["clean_stop_id"]))
    out = df.copy()
    out["clean_stop_id"] = out["stop_seq"].map(mapping)
    return out.dropna(subset=["clean_stop_id"]).assign(
        clean_stop_id=lambda x: x["clean_stop_id"].astype(int)
    )


print("=== Эталонные маршруты ===")
ref_routes = {}
for route in ROUTES:
    for direction in DIRECTIONS:
        key = (route, direction)
        ref_routes[key] = build_reference_route(df_clean, route, direction)

# Печатаем таблицу остановок для ручного заполнения STOP_ID_OVERRIDES (если нужно)
print("\n=== Список остановок для поиска пересечений ===")
for (route, direction), ref in ref_routes.items():
    if not ref.empty:
        dir_tag = "fwd" if direction == "Прямое" else "bwd"
        print(f"\n  {route}_{dir_tag}:")
        for _, row in ref.iterrows():
            print(f"    {int(row['clean_stop_id']):3d}  gid={int(row['global_stop_id'])}  {row['stop_name']}")


=== Эталонные маршруты ===
  Маршрут 20 (Прямое): 34 остановок (seq 0–33), global_id 20001..20034
  Маршрут 20 (Обратное): 33 остановок (seq 0–32), global_id 20001..20033
  Маршрут 48 (Прямое): 36 остановок (seq 0–35), global_id 48001..48036
  Маршрут 48 (Обратное): 35 остановок (seq 0–34), global_id 48001..48035
  Маршрут 55 (Прямое): 38 остановок (seq 1–38), global_id 55001..55038
  Маршрут 55 (Обратное): 37 остановок (seq 2–38), global_id 55001..55037

=== Список остановок для поиска пересечений ===

  20_fwd:
      1  gid=20001  Конечная станция "Пр. Культуры" (посадки и высадки нет)
      2  gid=20002  Конечная станция "Авангардная ул."
      3  gid=20003  Пр. Культуры, 26
      4  gid=20004  Пр. Просвещения
      5  gid=20005  Ул. Генерала Симоняка
      6  gid=20006  Универсам "Таллинский"
      7  gid=20007  Ул. Солдата Корзуна
      8  gid=20008  Ул. Руднева
      9  gid=20009  Торговый центр
     10  gid=20010  Ул. Лёни Голикова
     11  gid=20011  Ст. метро "Проспект Ветеран

In [15]:
def estimate_distances(df: pd.DataFrame, ref: pd.DataFrame) -> list:
    """
    Возвращает [[global_stop_id, dist_m], ...].
    Первая остановка — dist=0.
    """
    if ref.empty:
        return []

    d = attach_clean_stop_id(df, ref)
    d = d.sort_values(["trip_id", "arr_time"])
    d["time_diff_min"] = (d.groupby("trip_id")["arr_time"]
                           .diff()
                           .dt.total_seconds() / 60.0)

    # Словарь clean_stop_id → global_stop_id
    gid_map = dict(zip(ref["clean_stop_id"], ref["global_stop_id"]))
    n = len(ref)

    distances = [[gid_map[1], 0]]

    for stop_id in range(2, n + 1):
        stop_data = d[d["clean_stop_id"] == stop_id]["time_diff_min"]
        valid = stop_data[(stop_data >= 0.5) & (stop_data <= 15.0)]

        median_min = valid.median() if not valid.empty else 2.0
        dist_m = (median_min / 60.0) * ASSUMED_SPEED * 1000
        dist_m = max(100, int((dist_m // 50) * 50))
        distances.append([gid_map[stop_id], dist_m])

    return distances


In [16]:
def calculate_intensity(df: pd.DataFrame, ref: pd.DataFrame) -> list:
    """Возвращает [[global_stop_id, hour, intensity], ...]."""
    if ref.empty:
        return []

    d = attach_clean_stop_id(df, ref)
    gid_map = dict(zip(ref["clean_stop_id"], ref["global_stop_id"]))

    g = d.groupby(["clean_stop_id", "hour", "date_only"])["boarded"].sum().reset_index()
    g_sum  = g.groupby(["clean_stop_id", "hour"])["boarded"].sum()
    g_days = g.groupby(["clean_stop_id", "hour"])["date_only"].nunique()

    mask = g_days >= MIN_DAYS
    avg = (g_sum[mask] / g_days[mask]).fillna(0).round().astype(int).reset_index()
    avg.columns = ["clean_stop_id", "hour", "intensity"]

    result = [
        [gid_map[int(r.clean_stop_id)], int(r.hour), int(r.intensity)]
        for r in avg.itertuples(index=False)
    ]
    return sorted(result, key=lambda x: (x[0], x[1]))


def fill_intensity_gaps(intensity: list, stop_ids: list) -> list:
    """
    Заполняет дырки в часах — логика прежняя, но теперь итерируем
    по global_stop_ids, а не по range(1, N+1).
    """
    if not intensity:
        return []

    data_dict = {sid: {} for sid in stop_ids}
    for stop_id, hour, val in intensity:
        if stop_id in data_dict:
            data_dict[stop_id][hour] = val

    filled = []
    for stop_id in stop_ids:
        hours_data = data_dict[stop_id]

        if not hours_data:
            for h in range(24):
                filled.append([stop_id, h, 0])
            continue

        known_hours = sorted(hours_data.keys())
        first_h, last_h = known_hours[0], known_hours[-1]

        for h in range(24):
            if h in hours_data:
                val = hours_data[h]
            elif first_h < h < last_h:
                left_h  = max(kh for kh in known_hours if kh < h)
                right_h = min(kh for kh in known_hours if kh > h)
                ratio = (h - left_h) / (right_h - left_h)
                val = int(round(hours_data[left_h] + (hours_data[right_h] - hours_data[left_h]) * ratio))
            elif h < first_h:
                if first_h >= 6:
                    ratio = max(0.0, min(1.0, (h - 4) / (first_h - 4) if h >= 4 else 0))
                    val = int(round(hours_data[first_h] * ratio))
                else:
                    val = 0
            else:
                if last_h <= 22:
                    ratio = max(0.0, min(1.0, (23 - h) / (23 - last_h) if h <= 23 else 0))
                    val = int(round(hours_data[last_h] * ratio))
                else:
                    val = 0

            filled.append([stop_id, h, max(0, val)])

    return sorted(filled, key=lambda x: (x[0], x[1]))


def intensity_health(intensity: list, label: str = ""):
    total   = len(intensity)
    nonzero = sum(v[2] > 0 for v in intensity)
    pct     = (nonzero / total * 100) if total > 0 else 0
    print(f"  {label:30s} entries: {total:4d}, non-zero: {nonzero:4d} ({pct:.1f}%)")


In [17]:
def get_interval_at(t_min: float, route: str) -> int:
    """Возвращает интервал в минутах для времени t_min по расписанию маршрута."""
    sched = SCHEDULE_INTERVALS[route]
    for (from_min, to_min, interval) in sched["periods"]:
        if from_min <= t_min < to_min:
            return interval
    # до первого периода или после последнего — берём ближайший
    if t_min < sched["periods"][0][0]:
        return sched["periods"][0][2]
    return sched["periods"][-1][2]


def build_schedule(distances: list, route: str, stop_time: float,
                   acceleration_time: float, flow_speed: float) -> list:
    """
    Генерирует эталонное расписание — идеальные плановые времена прибытия
    на каждую остановку для каждого рейса.

    Физика: только distance + flow_speed + stop_time + acceleration_time.
    road_loads не используется — это эталон без задержек.

    Возвращает:
        [
          {
            "trip_id": 1,
            "departure_from_depot": 321.0,
            "stops": [[stop_id, arrival_min], ...]
          },
          ...
        ]
    """
    sched_cfg  = SCHEDULE_INTERVALS[route]
    start_min  = sched_cfg["start"]
    end_min    = sched_cfg["end"]

    # Предварительно считаем время хода между остановками (идеальное, без пробок)
    # distances = [[stop_id, dist_m], ...]  dist_m для первой остановки = 0
    # travel_time[i] — время от остановки i-1 до i (в минутах)
    speed_m_per_min = flow_speed * 1000 / 60.0
    leg_times = []  # leg_times[i] = время хода ДО i-й остановки
    for i, (stop_id, dist_m) in enumerate(distances):
        if i == 0:
            leg_times.append(0.0)
        else:
            leg_times.append(dist_m / speed_m_per_min)

    trips   = []
    trip_id = 1
    departure = start_min  # время отправления с первой остановки

    while departure <= end_min:
        stops = []
        current_time = departure

        for i, (stop_id, dist_m) in enumerate(distances):
            if i == 0:
                arrival = current_time
            else:
                arrival = current_time + leg_times[i] + acceleration_time
                current_time = arrival + stop_time  # стоянка на остановке

            stops.append([stop_id, round(arrival, 2)])

        trips.append({
            "trip_id":              trip_id,
            "departure_from_depot": round(departure - DEPOT_TO_FIRST_STOP_MIN, 2),
            "stops":                stops,
        })

        trip_id  += 1
        interval  = get_interval_at(departure, route)
        departure += interval

    return trips


In [18]:
def save_config_compact(config: dict, output_file: Path):
    """
    Записывает конфиг в читаемом формате.
    """
    stop_ids = config["stop_ids"]

    lines = ["{"]
    lines.append(f'  "route_id": {json.dumps(config["route_id"])},')
    lines.append(f'  "stop_ids": {json.dumps(stop_ids)},')
    lines.append(f'  "stop_number": {config["stop_number"]},')
    lines.append(f'  "distance": {json.dumps(config["distance"])},')

    # intensity — по одной остановке на строку
    lines.append('  "intensity": [')
    intensity_lines = []
    for sid in stop_ids:
        items = [x for x in config["intensity"] if x[0] == sid]
        if items:
            s = ", ".join(json.dumps(x) for x in items)
            intensity_lines.append(s)
    for i, text_line in enumerate(intensity_lines):
        comma = "," if i < len(intensity_lines) - 1 else ""
        lines.append(f"    {text_line}{comma}")
    lines.append("  ],")

    # schedule — один рейс на строку
    lines.append('  "schedule": [')
    schedule_list = config["schedule"]
    for i, trip in enumerate(schedule_list):
        comma = "," if i < len(schedule_list) - 1 else ""
        lines.append(f"    {json.dumps(trip)}{comma}")
    lines.append("  ],")

    lines.append(f'  "depot_to_first_stop": {config["depot_to_first_stop"]},')
    lines.append(f'  "min_rest_time": {config["min_rest_time"]},')
    lines.append(f'  "road_loads": {json.dumps(config["road_loads"])},')
    lines.append(f'  "flow_speed": {config["flow_speed"]},')
    lines.append(f'  "peak_stop": {config["peak_stop"]},')
    lines.append(f'  "tram_capacity": {config["tram_capacity"]},')
    lines.append(f'  "tram_count": {config["tram_count"]},')
    lines.append(f'  "simulation_hours": {config["simulation_hours"]},')
    lines.append(f'  "acceleration_time": {config["acceleration_time"]},')
    lines.append(f'  "stop_time": {config["stop_time"]},')
    lines.append(f'  "turnaround_time": {config["turnaround_time"]},')
    lines.append(f'  "target_utilization": {config["target_utilization"]},')
    lines.append(f'  "random_seed": {json.dumps(config["random_seed"])}')
    lines.append("}")

    output_file.write_text("\n".join(lines), encoding="utf-8")
    print(f"  Сохранено: {output_file.name}")


In [19]:
def build_config_for_route(df: pd.DataFrame, route: str, direction: str) -> dict | None:
    key = (route, direction)
    ref = ref_routes.get(key)

    if ref is None or ref.empty:
        print(f"  Нет эталона для {route} ({direction}), пропускаем")
        return None

    d = df[(df["route"] == route) & (df["direction"] == direction)]

    # global_stop_ids в порядке маршрута
    stop_ids    = ref["global_stop_id"].tolist()
    stop_number = len(stop_ids)

    distances        = estimate_distances(d, ref)
    raw_intensity    = calculate_intensity(d, ref)
    filled_intensity = fill_intensity_gaps(raw_intensity, stop_ids)

    schedule = build_schedule(
        distances         = distances,
        route             = route,
        stop_time         = 1.0,
        acceleration_time = 0.5,
        flow_speed        = float(ASSUMED_SPEED),
    )

    # peak_stop — середина маршрута (глобальный stop_id)
    peak_stop = stop_ids[stop_number // 2]

    # route_id: для прямого "20_fwd", для обратного "20_bwd"
    dir_tag  = "fwd" if direction == "Прямое" else "bwd"
    route_id = f"{route}_{dir_tag}"

    config = {
        "route_id":             route_id,
        "stop_ids":             stop_ids,
        "stop_number":          stop_number,
        "distance":             distances,
        "intensity":            filled_intensity,
        "schedule":             schedule,
        "depot_to_first_stop":  DEPOT_TO_FIRST_STOP_MIN,
        "min_rest_time":        MIN_REST_TIME_MIN,
        "road_loads": [
            [0, 0.1], [6, 0.4], [7, 0.7], [8, 0.9],
            [10, 0.6], [16, 0.7], [17, 0.9], [20, 0.4], [23, 0.1]
        ],
        "flow_speed":           int(ASSUMED_SPEED),
        "peak_stop":            peak_stop,
        "tram_capacity":        MAX_CAPACITY,
        "tram_count":           8,
        "simulation_hours":     24,
        "acceleration_time":    0.5,
        "stop_time":            1.0,
        "turnaround_time":      2.0,
        "target_utilization":   0.75,
        "random_seed":          42,
    }
    return config


print("=== Генерация конфигов ===\n")
all_configs = {}

for route in ROUTES:
    for direction in DIRECTIONS:
        print(f"Маршрут {route} ({direction}):")
        cfg = build_config_for_route(df_clean, route, direction)
        if cfg is None:
            continue

        intensity_health(cfg["intensity"], label=f"{route} {direction}")

        dir_tag  = "fwd" if direction == "Прямое" else "bwd"
        out_path = CONFIG_DIR / f"route_{route}_{dir_tag}_config.json"
        save_config_compact(cfg, out_path)
        all_configs[(route, direction)] = cfg
        print()


=== Генерация конфигов ===

Маршрут 20 (Прямое):
  20 Прямое                      entries:  816, non-zero:  606 (74.3%)
  Сохранено: route_20_fwd_config.json

Маршрут 20 (Обратное):
  20 Обратное                    entries:  792, non-zero:  619 (78.2%)
  Сохранено: route_20_bwd_config.json

Маршрут 48 (Прямое):
  48 Прямое                      entries:  864, non-zero:  492 (56.9%)
  Сохранено: route_48_fwd_config.json

Маршрут 48 (Обратное):
  48 Обратное                    entries:  840, non-zero:  443 (52.7%)
  Сохранено: route_48_bwd_config.json

Маршрут 55 (Прямое):
  55 Прямое                      entries:  912, non-zero:  657 (72.0%)
  Сохранено: route_55_fwd_config.json

Маршрут 55 (Обратное):
  55 Обратное                    entries:  888, non-zero:  643 (72.4%)
  Сохранено: route_55_bwd_config.json



In [20]:
print("=" * 60)
print("ИТОГО")
print("=" * 60)

for (route, direction), cfg in all_configs.items():
    n_stops = cfg["stop_number"]
    n_int   = len(cfg["intensity"])
    nz      = sum(v[2] > 0 for v in cfg["intensity"])
    max_int = max((v[2] for v in cfg["intensity"]), default=0)
    avg_d   = np.mean([d[1] for d in cfg["distance"][1:]]) if n_stops > 1 else 0

    print(f"\n  {route} {direction:8s}:")
    print(f"    route_id:               {cfg['route_id']}")
    print(f"    Остановок:              {n_stops}")
    print(f"    stop_ids[0..2]:         {cfg['stop_ids'][:3]}...")
    print(f"    Ср. расстояние:         {avg_d:.0f} м")
    print(f"    Intensity entries:      {n_int}")
    print(f"    Non-zero intensity:     {nz} ({(nz/n_int*100 if n_int else 0):.1f}%)")
    print(f"    Макс. intensity/ч:      {max_int}")

# Проверяем пересечения stop_ids между маршрутами
all_stop_ids_flat = [sid for cfg in all_configs.values() for sid in cfg["stop_ids"]]
shared = {sid for sid, cnt in Counter(all_stop_ids_flat).items() if cnt > 1}
if shared:
    print(f"\n  ↔ Общие stop_ids (заданы через STOP_ID_OVERRIDES): {sorted(shared)}")
else:
    print("\n  ℹ Пересечений stop_ids нет. Если маршруты физически пересекаются,")
    print("    заполни STOP_ID_OVERRIDES в ячейке 1 и перезапусти ноутбук.")

print("\n" + "=" * 60)
print(f"Конфиги сохранены в: {CONFIG_DIR.resolve()}")
print("=" * 60)

# Команда для запуска симуляции
config_args = " ".join(
    str(CONFIG_DIR / f"route_{r}_{d}_config.json")
    for r in ROUTES for d in ["fwd", "bwd"]
)
print(f"\nЗапуск симуляции:")
print(f"  python -m simulation.runner --configs {config_args}")


ИТОГО

  20 Прямое  :
    route_id:               20_fwd
    Остановок:              34
    stop_ids[0..2]:         [20001, 20002, 20003]...
    Ср. расстояние:         412 м
    Intensity entries:      816
    Non-zero intensity:     606 (74.3%)
    Макс. intensity/ч:      24

  20 Обратное:
    route_id:               20_bwd
    Остановок:              33
    stop_ids[0..2]:         [20001, 20002, 20003]...
    Ср. расстояние:         414 м
    Intensity entries:      792
    Non-zero intensity:     619 (78.2%)
    Макс. intensity/ч:      29

  48 Прямое  :
    route_id:               48_fwd
    Остановок:              36
    stop_ids[0..2]:         [48001, 48002, 48003]...
    Ср. расстояние:         419 м
    Intensity entries:      864
    Non-zero intensity:     492 (56.9%)
    Макс. intensity/ч:      14

  48 Обратное:
    route_id:               48_bwd
    Остановок:              35
    stop_ids[0..2]:         [48001, 48002, 48003]...
    Ср. расстояние:         404 м
    Inten